# Prepare TimeXNet data - February 5, 2016
Create lists of early, intermediate, and late genes along with their scores.  Format the PPI network.

In [1]:
import pandas as pd
import os, math, shutil

## Configure directories

In [2]:
phospho_file = os.path.join('..','..','phosphorylation-data-sets','McKenna2014','aggregated_replicateABC_tukey_orig_firstprev.txt')
map_file = os.path.join('..','..','phosphorylation-data-sets','McKenna2014','McKenna2014_replicate_C_uniprotAccession_mapped.txt')
ppi_file = os.path.join('..','..', 'interactomes', 'PhosphoSite_iRefIndex13.0_uniprot_overwrite_pcsf.txt')
txn_dir = os.path.join('..','..','TimeXNet','data')
remote_data_dir = os.path.join('U:','home','agitter','projects','egfr','TimeXNet','data')

## Create helper function
Identifies whether a peptide is early, intermediate, or late based on the time point of its minimum p-value
* tp2 (2 min), tp3 (4 min): Early
* tp4 (8 min), tp5 (16 min), tp6 (32 min): Intermediate
* tp7 (64 min), tp8 (128 min): Late

In [3]:
def time_interval(row):
    # Track the min p-value in each interval
    early = min(row['tp2'], row['tp3'])
    inter = min(row['tp4'], row['tp5'], row['tp6'])
    late = min(row['tp7'], row['tp8'])
    
    # For testing
    #print('E:%f\tI:%f\tL:%f' % (early, inter, late))
    
    if early <= inter and early <= late:
        return 'Early'
    elif inter <= early and inter <= late:
        return 'Intermediate'
    elif late <= inter and late <= early:
        return 'Late'
    else:
        raise RuntimeError('Timing must be early, intermediate, or late')

## Load the EGFR phosphorylation data
Use the version that has p-values at each time point that are the minimum from the first and previous time point comparisons from Tukey HSD test.

In [4]:
phospho_df = pd.read_csv(phospho_file, sep='\t')
# Store the first protein id, which is considered to be the best match
phospho_df['first.protein'] = phospho_df.apply(lambda row: row['protein'].upper().split(',')[0], axis=1)
phospho_df.head()

,peptide,protein,gene.name,min.tukey.pval,neg.log10.min.tukey.pval,rep1.tp0,rep1.tp2,rep1.tp4,rep1.tp8,rep1.tp16,...,tp8,median.tp0,median.tp2,median.tp4,median.tp8,median.tp16,median.tp32,median.tp64,median.tp128,first.protein
0,-.n[305.21]MRK[432.30]ET[181.01]PPPLVPPAAR.E,"Q9BQA1,B4DJP0,B4DP38",WDR77,0.241322,0.617402,3.266944,2.848139,2.580626,3.058835,2.762461,...,0.376328,2.76679,2.68569,2.42117,2.81289,2.12906,2.89894,2.79520,3.11382,Q9BQA1
1,-.n[305.21]MSTTAGY[243.03]LARR.A,Q96SA0,Putative uncharacterized protein,0.115603,0.937030,5.606161,4.002793,3.587753,3.832577,4.777530,...,0.115603,7.69423,3.99119,3.82037,4.34460,4.75965,5.04384,5.77301,8.24482,Q96SA0
2,K.n[305.21]AAASGQPRPEMQC[160.03]PAEHEEDMY[243....,Q96BN8,FAM105B,0.005700,2.244160,0.099820,0.861090,0.568702,0.498091,0.364445,...,0.999629,0.10590,0.86109,0.56870,0.49809,0.36445,0.23328,0.15722,0.11192,Q96BN8
3,K.n[305.21]AAGPAS[167.00]PPEEDPER.T,Q96KM6,ZNF512B,0.820275,0.086041,0.369111,0.760905,0.535401,0.586129,0.503314,...,0.997236,0.70996,0.76090,0.74556,0.58613,0.65659,0.56510,0.70899,0.59856,Q96KM6
4,K.n[305.21]AALGLQDS[167.00]DDEDAAVDIDEQIESMFNS...,"Q7L014,A8K6X3,D3DQA6",DDX46,0.009719,2.012386,5.980349,5.078731,4.807882,4.988418,4.692977,...,0.091643,5.77342,5.13979,4.39768,4.98842,4.58252,4.63332,4.95409,5.95735,Q7L014


## Inspect the first protein id extraction

In [5]:
phospho_df[['protein','first.protein']]

,protein,first.protein
0,"Q9BQA1,B4DJP0,B4DP38",Q9BQA1
1,Q96SA0,Q96SA0
2,Q96BN8,Q96BN8
3,Q96KM6,Q96KM6
4,"Q7L014,A8K6X3,D3DQA6",Q7L014
5,Q96MW1,Q96MW1
6,"P50914,A8K3Q9,A8K7N0,Q6IPH7",P50914
7,B4DSL6,B4DSL6
8,"Q8IYB3,A9Z1X7,B7Z7U0,E9PCT1",Q8IYB3
9,"Q96ST2,B4DGM5,B4DJ84,B4DL52",Q96ST2


## Check helper function
Uncomment the print statement in the helper function

In [6]:
for i in range(10):
    row = phospho_df.ix[i]
    print row['tp2':'tp8']
    print time_interval(row)
    print '\n'

tp2    0.9998113
tp3    0.4849091
tp4    0.5712768
tp5    0.3749091
tp6    0.2413225
tp7    0.9030822
tp8    0.3763276
Name: 0, dtype: object
Intermediate


tp2    0.2079877
tp3    0.2121369
tp4    0.2962231
tp5    0.3746407
tp6    0.6958744
tp7     0.983827
tp8    0.1156033
Name: 1, dtype: object
Late


tp2    0.005699539
tp3      0.2594882
tp4      0.3897096
tp5      0.7468051
tp6      0.9900425
tp7      0.9994365
tp8      0.9996289
Name: 2, dtype: object
Early


tp2    0.9999947
tp3    0.9999964
tp4    0.9861355
tp5    0.8202747
tp6    0.9633311
tp7    0.9956128
tp8    0.9972364
Name: 3, dtype: object
Intermediate


tp2      0.7386493
tp3     0.01682738
tp4      0.1893444
tp5    0.009718833
tp6      0.2184715
tp7      0.4998588
tp8     0.09164338
Name: 4, dtype: object
Intermediate


tp2     0.8829984
tp3     0.9944826
tp4     0.4679519
tp5     0.7198828
tp6     0.9999948
tp7     0.9999999
tp8    0.07655296
Name: 5, dtype: object
Late


tp2      0.9695877
tp3    0.009936292
tp4     

## Map each protein to its most significant peptide
Identify the most significant temporal interval for each peptide.  Then select the time point corresponding to that peptide

In [7]:
phospho_df['interval'] = phospho_df.apply(lambda row: time_interval(row), axis=1)

# first.protein is the key in both dicts
pval_map = dict()
interval_map = dict()

for index, row in phospho_df.iterrows():
    prot = row['first.protein']
    pval = row['min.tukey.pval']
    interval = row['interval']
    
    if (not prot in pval_map) or (pval < pval_map[prot]):
        pval_map[prot] = pval
        interval_map[prot] = interval

print 'Found %d unique proteins' % len(pval_map)


Found 701 unique proteins


## Load the UniProt id mapping

In [8]:
uniprot_df = pd.read_csv(map_file, sep='\t')
uniprot_map = dict(zip(uniprot_df['From'],uniprot_df['To']))

print 'UniProt ids for %d proteins loaded' % len(uniprot_df)
uniprot_df.head()

UniProt ids for 6091 proteins loaded


,From,To
0,A8K1N6,A8K1N6_HUMAN
1,B2R5V7,B2R5V7_HUMAN
2,B4DSD6,B4DSD6_HUMAN
3,H7C127,H7C127_HUMAN
4,H7C128,H7C128_HUMAN


## Write a file for each of the intervals
Convert the p-values to -log_10(p-value).  Deleted UniProt ids are mapped to ID_PSEUDONODE. These are different than the pseudonodes used with TPS, which used the peptide sequence, but it doesn't matter because neither id maps to the PPI network and there won't be collisions.

In [9]:
early_file = os.path.join(txn_dir,'egfr_early_proteins.txt')
intermediate_file = os.path.join(txn_dir,'egfr_intermediate_proteins.txt')
late_file = os.path.join(txn_dir,'egfr_late_proteins.txt')

early_count = 0
intermediate_count = 0
late_count = 0

with open(early_file, 'w') as early_f:
    with open(intermediate_file, 'w') as intermediate_f:
        with open(late_file, 'w') as late_f:
            # Loop through all proteins and write to the appropriate file
            for prot in sorted(pval_map.keys()):
                if not prot in uniprot_map:
                    print 'Creating pseudonode for UniProt id %s' % prot
                    uniprot_map[prot] = '%s_PSEUDONODE' % prot
                    
                interval = interval_map[prot]
                line = '%s\t%f\n' % (uniprot_map[prot], -math.log10(pval_map[prot]))
                if interval == 'Early':
                    early_f.write(line)
                    early_count += 1
                elif interval == 'Intermediate':
                    intermediate_f.write(line)
                    intermediate_count += 1
                elif interval == 'Late':
                    late_f.write(line)
                    late_count += 1
                else:
                    raise RuntimeError('Timing must be early, intermediate, or late')

print 'Early: %d\nIntermediate: %d\nLate: %d' % (early_count, intermediate_count, late_count)

Creating pseudonode for UniProt id H0YAM2
Early: 306
Intermediate: 245
Late: 150


## Convert the PPI network to the TimeXNet format

In [10]:
ppi_count = 0
with open(os.path.join(txn_dir,'PhosphoSite_iRefIndex13.0_uniprot_overwrite_txn.txt'), 'w') as out_f:
    with open(ppi_file) as in_f:
        for line in in_f:
            parts = line.strip().split('\t')
            p1 = parts[0]
            p2 = parts[1]
            weight = parts[2]
            direc = parts[3]
            
            if direc == 'D':
                out_direc = '->'
            elif direc == 'U':
                out_direc = '--'
            else:
                raise RuntimeError('Unrecognized direction: %s' % direc)
            
            out_f.write('%s\t%s\t%s\t%s\n' % (p1, p2, out_direc, weight))
            ppi_count += 1
            
print 'Converted %d PPI' % ppi_count            

Converted 161901 PPI


## Copy data files to the WID server

In [11]:
filenames = os.listdir(txn_dir)
for fn in filenames:
    print 'Copy %s to server' % fn
    shutil.copy2(os.path.join(txn_dir, fn), os.path.join(remote_data_dir, fn))

Copy egfr_early_proteins.txt to server
Copy egfr_intermediate_proteins.txt to server
Copy egfr_late_proteins.txt to server
Copy PhosphoSite_iRefIndex13.0_uniprot_overwrite_txn.txt to server
